# Compass Analysis via SVD Archetype Space

This notebook implements a targeted approach to analysis of characters along a particular set of traits using a pre-computed Singular Value Decomposition (SVD) of character-trait data.

## Approach:
1. **Trait Selection**: Decide on "core" traits for your axis.
2. **Moral Vector ($A_m$)**: Create a vector in the trait space where only our chosen core traits are active.
3. **Archetype Space Projection**: Compute the chosen axis in archetype space using $U^T \times A_m$.
4. **Character Projection**: Project characters onto this axis to find characters most aligned / unaligned with your axis.
5. **Trait Alignment**: Find which other traits align with this axis.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set paths
data_path = 'data/plain/current/2000/'

def load_matrix(filename):
    return np.loadtxt(os.path.join(data_path, filename))

print("Loading data...")
traits_df = pd.read_csv(os.path.join(data_path, 'data_traits.tsv'), sep='\t')
characters_df = pd.read_csv(os.path.join(data_path, 'data_characters.tsv'), sep='\t')

U = load_matrix('data_archetype_space_U.txt')
Sigma_vec = load_matrix('data_archetype_space_Sigma.txt')
V = load_matrix('data_archetype_space_V.txt')

# Ensure dimensions match U (464 traits)
k = U.shape[0]
Sigma_vec = Sigma_vec[:k]
V = V[:, :k]

Sigma = np.diag(Sigma_vec)

print(f"U shape: {U.shape}")
print(f"Sigma shape: {Sigma.shape}")
print(f"V shape: {V.shape}")
print(f"Traits: {len(traits_df)}, Characters: {len(characters_df)}")

Loading data...
U shape: (464, 464)
Sigma shape: (464,)
V shape: (2000, 464)
Traits: 464, Characters: 2000


## 1. Select Traits

You do this by making a config file with the traits you consider "core". See examples. Currently you only have to specify the trait you WANT out of the differential. This breaks for a few traits like "warm" (which is included in more than one differential. 

Currently, all traits are weighted equally.

Note: In this dataset, the $A$ matrix is typically constructed such that the `right pole` is positive and the `left pole` is negative. 
We will check the poles for each trait to ensure we assign the correct weight (e.g., +1 or -1) to represent the "Good" side.

In [2]:
# choose a config file to import traits from
from config.trait_config_hobbit import core_traits

right_pole_matches = traits_df[traits_df['right pole'].isin(core_traits)][['index', 'differential']]
right_pole_matches['weight'] = 1

left_pole_matches = traits_df[traits_df['left pole'].isin(core_traits)][['index', 'differential']]
left_pole_matches['weight'] = -1

matches = pd.concat([right_pole_matches, left_pole_matches])
matches

,index,differential,weight
68,69,tall :: short,1
132,133,adventurous :: stick-in-the-mud,1
220,221,wholesome :: salacious,-1


In [3]:
Am = np.zeros((U.shape[0], 1))
for idx, data in matches.iterrows():
    # Subtract 1 because index in TSV is 1-based
    print(data['weight'])
    Am[idx-1] = data['weight']

print(f"Moral vector Am created with {matches.shape[0]} traits.")

1
1
-1
Moral vector Am created with 3 traits.


## 2. Project Morality into Archetype Space

The moral axis in the low-dimensional archetype space is given by $w_m = U^T A_m$.

In [4]:
wm = U.T @ Am
print(f"Moral axis wm shape: {wm.shape}")

Moral axis wm shape: (464, 1)


## 3. Project Characters onto the Moral Axis

Character coordinates in archetype space are $C = \Sigma V^T$.
The projection (score) of characters onto our moral axis is $S_{chars} = C^T w_m = (V \Sigma) w_m$.

In [5]:
print("V: ", V.shape)
print("Sigma: ", Sigma.shape)
print("wm: ", wm.shape)

C = Sigma @ V.T
print("Sigma @ V.T:", C.shape)
print("C.T: ", C.T.shape)
character_scores = C.T @ wm


# character_scores = (V @ Sigma) @ wm


characters_df['alignment_score'] = character_scores

print("Top 10 Aligned Characters:")
display(characters_df.sort_values('alignment_score', ascending=False).head(10)[['character', 'character/story', 'alignment_score']])

print("\nTop 10 Misaligned Characters:")
display(characters_df.sort_values('alignment_score', ascending=True).head(10)[['character', 'character/story', 'alignment_score']])

V:  (2000, 464)
Sigma:  (464,)
wm:  (464, 1)
Sigma @ V.T: (2000,)
C.T:  (2000,)


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 464 is different from 2000)

## 4. Find Other-Aligned Traits

Which other traits (out of the 464) are most aligned with this axis?
The trait alignment scores are $S_{traits} = U \Sigma^2 w_m$.

In [6]:
trait_scores = U @ (Sigma_vec**2 * wm.flatten())
traits_df['moral_alignment'] = trait_scores

print("Traits most aligned with 'Good':")
display(traits_df.sort_values('moral_alignment', ascending=False).head(15)[['differential', 'moral_alignment']])

print("\nTraits most aligned with 'Evil':")
display(traits_df.sort_values('moral_alignment', ascending=True).head(15)[['differential', 'moral_alignment']])

ValueError: operands could not be broadcast together with shapes (464,2000) (464,) 

## 5. Moral Duality Exploration

Let's add a second axis (e.g., Masculine/Feminine) to create a 2D moral compass.
We can then see how characters cluster or spread.

In [ ]:
# Select a duality trait
# Index 4: masculine :: feminine
duality_idx = 4
Ad = np.zeros((U.shape[0], 1))
Ad[duality_idx-1] = 1 # Positive = Feminine

wd = U.T @ Ad
duality_scores = (V @ Sigma) @ wd
characters_df['duality_score'] = duality_scores

plt.figure(figsize=(12, 10))
sns.scatterplot(data=characters_df, x='moral_score', y='duality_score', alpha=0.5)

# Label some outliers
top_chars = characters_df.sort_values('moral_score', ascending=False).head(5)
bot_chars = characters_df.sort_values('moral_score', ascending=True).head(5)
extreme_duality = characters_df.sort_values('duality_score', ascending=False).head(5)
extreme_neg_duality = characters_df.sort_values('duality_score', ascending=True).head(5)

for i, row in pd.concat([top_chars, bot_chars, extreme_duality, extreme_neg_duality]).iterrows():
    plt.text(row['moral_score'], row['duality_score'], row['character'], fontsize=9)

plt.axvline(0, color='grey', linestyle='--')
plt.axhline(0, color='grey', linestyle='--')
plt.xlabel('Moral Score (Evil -> Good)')
plt.ylabel('Duality Score (Masculine -> Feminine)')
plt.title('2D Moral Compass: Morality vs Gender Duality')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Clustering Moral "Types"

Finally, we can cluster characters based on their projection into a multi-dimensional "moral space" (using more moral traits or just the top components of morality).

In [ ]:
from sklearn.cluster import KMeans

# Let's use the moral score and a few other moral-aligned dimensions
moral_features = characters_df[['moral_score', 'duality_score']]

kmeans = KMeans(n_clusters=4, random_state=42)
characters_df['moral_cluster'] = kmeans.fit_predict(moral_features)

plt.figure(figsize=(10, 8))
sns.scatterplot(data=characters_df, x='moral_score', y='duality_score', hue='moral_cluster', palette='viridis', alpha=0.6)
plt.title('Character Clusters in Moral-Duality Space')
plt.show()